# 00-02 텐서 기초

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 텐서 만들기

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])  # (3,)
b = torch.tensor([[1, 2, 3], [4, 5, 6]])  # (2, 3)
print(a)
print(b)
print(a.shape, b.shape)
print(a.dtype, b.dtype)

In [ ]:
zeros = torch.zeros(2, 3)  # (2, 3)
ones = torch.ones(2, 3)  # (2, 3)
rand = torch.rand(2, 3)  # (2, 3), 0~1 사이 무작위 값
seq = torch.arange(6)  # (6,)
print(zeros)
print(rand)
print(seq)

In [ ]:
x = torch.arange(12).reshape(3, 4)  # (3, 4)

plt.figure(figsize=(4, 3))
plt.imshow(x, cmap="Blues")
for i in range(3):
    for j in range(4):
        plt.text(j, i, int(x[i, j]), ha="center", va="center")
plt.xlabel("dim 1 (4)")
plt.ylabel("dim 0 (3)")
plt.title("shape (3, 4)")
plt.colorbar()
plt.show()

## 2. 모양 바꾸기

In [ ]:
x = torch.arange(12)  # (12,)
print(x.reshape(3, 4).shape)  # (3, 4)
print(x.reshape(2, 2, 3).shape)  # (2, 2, 3)
print(x.reshape(4, -1).shape)  # (4, 3), -1은 남는 크기를 자동으로 계산해요

In [ ]:
m = torch.arange(6).reshape(2, 3)  # (2, 3)
print(m.T.shape)  # (3, 2)
print(m.unsqueeze(0).shape)  # (1, 2, 3)
print(m.unsqueeze(0).squeeze(0).shape)  # (2, 3)

## 3. 인덱싱

In [ ]:
m = torch.arange(12).reshape(3, 4)  # (3, 4)
print(m[0])  # 첫 번째 행, (4,)
print(m[:, 1])  # 두 번째 열, (3,)
print(m[1:, 2:])  # (2, 2)
print(m[m > 6])  # 조건에 맞는 값만, (5,)

## 4. 연산

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # (2, 2)
print(x + 10)
print(x * x)  # 같은 위치끼리 곱하기
print(x.sum(), x.mean())
print(x.sum(dim=0))  # 0번 축을 따라 더하기, (2,)
print(x.sum(dim=1))  # 1번 축을 따라 더하기, (2,)

In [ ]:
A = torch.rand(4, 3)  # (4, 3)
B = torch.rand(3, 2)  # (3, 2)
C = A @ B  # (4, 2)
print(C.shape)

## 5. 브로드캐스팅

In [ ]:
x = torch.zeros(4, 3)  # (4, 3)
bias = torch.tensor([1.0, 2.0, 3.0])  # (3,)
print(x + bias)  # bias가 4개 행에 똑같이 더해져요, (4, 3)

## 6. GPU로 옮기기

In [ ]:
x = torch.rand(3, 3)  # (3, 3)
print(x.device)
x_gpu = x.to(device)
print(x_gpu.device)

In [ ]:
y = torch.rand(3, 3)  # CPU에 있는 텐서
try:
    x_gpu + y
    print("같은 장치에 있어서 계산돼요")
except RuntimeError as e:
    print("오류:", str(e)[:80])

In [ ]:
import time

def time_matmul(dev, n=2048, repeat=10):
    a = torch.rand(n, n, device=dev)  # (n, n)
    b = torch.rand(n, n, device=dev)  # (n, n)
    a @ b  # 첫 실행은 준비 시간이 섞여서 한 번 버려요
    if dev == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeat):
        a @ b
    if dev == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) / repeat

print(f"CPU: {time_matmul('cpu') * 1000:.1f} ms")
if device == "cuda":
    print(f"GPU: {time_matmul('cuda') * 1000:.1f} ms")

## 7. 파이썬 값과 NumPy로 바꾸기

In [ ]:
s = torch.tensor([1.0, 2.0, 3.0]).sum()
print(s, s.item())

g = torch.rand(2, 2).to(device)
print(g.cpu().numpy())